# Zero-Order Optimization for Logistic Regression

This notebook demonstrates a **zero-order (derivative-free) gradient estimator** applied to minimizing a logistic regression loss. The key idea: instead of computing the true gradient $\nabla f(x)$, we approximate it using only function evaluations along random directions.

---

### Setup

We minimize the logistic loss (with optional L2 regularization):

$$
f(x) = \frac{1}{n} \sum_{i=1}^{n} \log\bigl(1 + e^{-y_i \cdot (A_i^\top x)}\bigr) + \frac{\lambda}{2} \|x\|^2
$$

### Zero-Order Gradient Estimator

$$
g(x) = \frac{1}{B} \sum_{i=1}^{B} \frac{f(x + \mu \, u_i) - f(x)}{\mu} \cdot u_i
$$

where $u_i \sim \mathcal{U}(\mathbb{S}^{d-1})$ are uniform random unit vectors, $\mu > 0$ is the smoothing parameter, and $B$ is the batch size.

### Update Rule

$$
x_{k+1} = x_k - \eta \cdot g(x_k)
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import Optional

# Reproducibility
np.random.seed(42)

print("Libraries loaded successfully.")

## 1. Hyperparameters

Adjust the parameters below to explore different regimes.

In [ ]:
# ── Problem dimensions ──────────────────────────────────────────────────────
dim        = 10       # feature dimension (bias column will be appended → d = dim+1)
n          = 200      # number of data points

# ── Algorithm hyperparameters ────────────────────────────────────────────────
lr         = 0.5      # learning rate  η
mu         = 0.1      # smoothing / finite-difference step  μ
batch_size = 20       # number of random directions per step  B
n_steps    = 300      # total number of gradient steps

# ── Regularization ───────────────────────────────────────────────────────────
lam        = 1e-2     # L2 regularization coefficient λ  (set to 0.0 to disable)

print(f"Problem: n={n} samples, dim={dim} features  →  parameter vector size d={dim+1} (with bias)")
print(f"Algorithm: lr={lr}, mu={mu}, batch_size={batch_size}, steps={n_steps}, lambda={lam}")

## 2. Data Generation

We generate a random binary classification dataset:
- Feature matrix $A_\text{raw} \in \mathbb{R}^{n \times d}$ with entries drawn from $\mathcal{N}(0,1)$.
- Labels $y \in \{-1, +1\}^n$ generated from a ground-truth weight vector $x^\star$.
- A bias column of ones is appended so the model can learn an intercept.

In [ ]:
def generate_data(n: int, dim: int, noise: float = 0.1):
    """
    Generate a synthetic binary classification dataset.

    Returns
    -------
    A : np.ndarray, shape (n, dim+1)
        Feature matrix with an appended bias column of ones.
    y : np.ndarray, shape (n,)
        Labels in {-1, +1}.
    x_star : np.ndarray, shape (dim+1,)
        Ground-truth weight vector used to generate the labels.
    """
    # Raw features
    A_raw = np.random.randn(n, dim)

    # Append bias column (all ones)
    A = np.hstack([A_raw, np.ones((n, 1))])   # shape (n, dim+1)

    # Ground-truth parameters (sparse for clarity)
    x_star = np.random.randn(dim + 1)
    x_star /= np.linalg.norm(x_star)          # unit norm

    # Logistic labels with optional label noise
    logits = A @ x_star
    probs  = 1.0 / (1.0 + np.exp(-logits))
    y_raw  = (np.random.rand(n) < probs).astype(float)   # {0, 1}

    # Flip noise fraction of labels
    flip_mask = np.random.rand(n) < noise
    y_raw[flip_mask] = 1.0 - y_raw[flip_mask]

    y = 2.0 * y_raw - 1.0    # {-1, +1}
    return A, y, x_star


A, y, x_star = generate_data(n, dim)
d = A.shape[1]    # = dim + 1

print(f"Feature matrix A : shape {A.shape}  (bias column appended ✓)")
print(f"Label vector y   : shape {y.shape},  unique values = {np.unique(y)}")
print(f"Ground-truth x*  : shape {x_star.shape},  ||x*|| = {np.linalg.norm(x_star):.4f}")

## 3. Loss Function

### Logistic Loss (with L2 regularization)

$$
f(x) = \frac{1}{n}\sum_{i=1}^{n} \log\!\left(1 + e^{-y_i (A_i^\top x)}\right) + \frac{\lambda}{2}\|x\|^2
$$

We use the numerically stable form $\log(1 + e^{-t}) = \log(1 + e^{-|t|}) + \max(0, -t)$ to avoid overflow.

In [ ]:
def logistic_loss(
    x: np.ndarray,
    A: np.ndarray,
    y: np.ndarray,
    lam: float = 0.0,
) -> float:
    """
    Compute the logistic loss (with optional L2 regularization).

    Parameters
    ----------
    x   : parameter vector, shape (d,)
    A   : feature matrix,   shape (n, d)   — bias column already included
    y   : labels {-1,+1},   shape (n,)
    lam : L2 regularization coefficient λ ≥ 0

    Returns
    -------
    float : f(x)
    """
    margins = y * (A @ x)                          # shape (n,)
    # Numerically stable log(1 + exp(-t))
    loss = np.mean(np.log1p(np.exp(-margins)))
    if lam > 0:
        loss += 0.5 * lam * np.dot(x, x)
    return float(loss)


# Quick sanity check
x0 = np.zeros(d)
print(f"Loss at zero vector  : {logistic_loss(x0, A, y, lam):.6f}")
print(f"Loss at ground truth : {logistic_loss(x_star, A, y, lam):.6f}")
print(f"Loss at 5×ground truth: {logistic_loss(5*x_star, A, y, lam):.6f}  ← should be lower if λ=0")

## 4. Zero-Order Gradient Estimator & Optimizer

### Random Unit Vectors
We sample $u_i \sim \mathcal{N}(0, I_d)$ then normalize: $u_i \leftarrow u_i / \|u_i\|$, giving uniform directions on $\mathbb{S}^{d-1}$.

### Gradient Estimator
$$
g(x) = \frac{1}{B}\sum_{i=1}^{B} \frac{f(x + \mu u_i) - f(x)}{\mu} \cdot u_i
$$

This is an unbiased estimator of $\nabla f_\mu(x)$, the gradient of the **Gaussian-smoothed** surrogate $f_\mu$.

In [ ]:
def sample_unit_vectors(batch_size: int, d: int) -> np.ndarray:
    """
    Sample `batch_size` uniform random unit vectors on S^{d-1}.

    Strategy: draw from N(0, I_d) then normalize each row.

    Returns
    -------
    U : np.ndarray, shape (batch_size, d)
    """
    U = np.random.randn(batch_size, d)              # Gaussian samples
    norms = np.linalg.norm(U, axis=1, keepdims=True)
    return U / norms                                 # normalize to unit sphere


def zero_order_gradient(
    x: np.ndarray,
    A: np.ndarray,
    y: np.ndarray,
    mu: float,
    batch_size: int,
    lam: float = 0.0,
    fx: Optional[float] = None,
) -> tuple[np.ndarray, float]:
    """
    Compute the zero-order gradient estimate g(x).

    Parameters
    ----------
    x          : current parameter vector, shape (d,)
    A, y       : dataset
    mu         : finite-difference step size
    batch_size : number of random directions B
    lam        : L2 coefficient
    fx         : f(x) pre-computed (to save one evaluation); computed if None

    Returns
    -------
    g  : gradient estimate, shape (d,)
    fx : f(x)  (returned for reuse in the main loop)
    """
    if fx is None:
        fx = logistic_loss(x, A, y, lam)

    d = len(x)
    U = sample_unit_vectors(batch_size, d)          # (B, d)

    # Perturbed losses: f(x + mu * u_i) for each i
    X_pert = x[None, :] + mu * U                   # (B, d)
    f_pert = np.array([
        logistic_loss(X_pert[i], A, y, lam)
        for i in range(batch_size)
    ])                                               # (B,)

    # Finite-difference coefficients
    coeffs = (f_pert - fx) / mu                     # (B,)

    # Gradient estimate: weighted average of directions
    g = (coeffs[:, None] * U).mean(axis=0)          # (d,)
    return g, fx


# Quick check: gradient should point away from zero (loss decreases along -g)
g_test, fx_test = zero_order_gradient(x0, A, y, mu=0.05, batch_size=50, lam=lam)
print(f"Zero-order gradient norm at x=0 : {np.linalg.norm(g_test):.4f}")
print(f"f(x=0)                          : {fx_test:.6f}")

In [ ]:
def zero_order_gd(
    x_init: np.ndarray,
    A: np.ndarray,
    y: np.ndarray,
    lr: float,
    mu: float,
    batch_size: int,
    n_steps: int,
    lam: float = 0.0,
    verbose: bool = True,
    log_every: int = 50,
) -> dict:
    """
    Zero-order gradient descent.

    Update rule:
        x_{k+1} = x_k - lr * g(x_k)

    Tracking per step:
        - loss f(x_k)
        - ||x_k||  (parameter norm)
        - ||g(x_k)||  (gradient estimate norm)
        - ||x_k - x*||  (distance to ground truth)

    Returns
    -------
    history : dict with keys 'loss', 'param_norm', 'grad_norm', 'dist_to_xstar'
    x_final : np.ndarray
    """
    x = x_init.copy()
    history = {
        "loss"         : [],
        "param_norm"   : [],
        "grad_norm"    : [],
        "dist_to_xstar": [],
    }

    for k in range(n_steps):
        # ── Zero-order gradient estimate ────────────────────────────────────
        g, fx = zero_order_gradient(x, A, y, mu, batch_size, lam)

        # ── Tracking ────────────────────────────────────────────────────────
        history["loss"].append(fx)
        history["param_norm"].append(np.linalg.norm(x))
        history["grad_norm"].append(np.linalg.norm(g))
        history["dist_to_xstar"].append(np.linalg.norm(x - x_star))

        if verbose and (k % log_every == 0 or k == n_steps - 1):
            print(
                f"Step {k:4d} | loss = {fx:.5f} | "
                f"||x|| = {history['param_norm'][-1]:.4f} | "
                f"||g|| = {history['grad_norm'][-1]:.4f} | "
                f"||x-x*|| = {history['dist_to_xstar'][-1]:.4f}"
            )

        # ── Parameter update ────────────────────────────────────────────────
        x = x - lr * g

    return history, x

print("Optimizer defined ✓")

## 5. Run the Optimizer

In [ ]:
np.random.seed(0)
x_init = np.zeros(d)    # start from the origin

print("Starting Zero-Order Gradient Descent...")
print("=" * 70)

history, x_final = zero_order_gd(
    x_init    = x_init,
    A         = A,
    y         = y,
    lr        = lr,
    mu        = mu,
    batch_size= batch_size,
    n_steps   = n_steps,
    lam       = lam,
    verbose   = True,
    log_every = max(1, n_steps // 10),
)

print("=" * 70)
print(f"\nFinal loss         : {history['loss'][-1]:.6f}")
print(f"Initial loss       : {history['loss'][0]:.6f}")
print(f"Loss reduction     : {history['loss'][0] - history['loss'][-1]:.6f}")
print(f"||x_final - x*||   : {history['dist_to_xstar'][-1]:.6f}")

## 6. Visualization

In [ ]:
steps = np.arange(n_steps)

fig = plt.figure(figsize=(14, 10))
fig.suptitle(
    f"Zero-Order GD  |  dim={dim}, n={n}, lr={lr}, μ={mu}, B={batch_size}, λ={lam}",
    fontsize=13, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

# ── 1. Training Loss ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(steps, history["loss"], color="#2563EB", lw=1.8, label="f(xₖ)")
ax1.set_title("Training Loss", fontweight="bold")
ax1.set_xlabel("Step k")
ax1.set_ylabel("f(xₖ)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# ── 2. Log-scale Loss ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
loss_arr = np.array(history["loss"])
loss_min  = loss_arr.min()
shifted   = loss_arr - loss_min + 1e-12   # avoid log(0)
ax2.semilogy(steps, shifted, color="#DC2626", lw=1.8)
ax2.set_title("Loss – min (log scale)", fontweight="bold")
ax2.set_xlabel("Step k")
ax2.set_ylabel("f(xₖ) – f*  (log)")
ax2.grid(True, alpha=0.3, which="both")

# ── 3. Parameter Norm ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(steps, history["param_norm"], color="#16A34A", lw=1.8, label="||xₖ||")
ax3.axhline(np.linalg.norm(x_star), color="gray", ls="--", lw=1.2, label="||x*||")
ax3.set_title("Parameter Norm", fontweight="bold")
ax3.set_xlabel("Step k")
ax3.set_ylabel("||xₖ||₂")
ax3.grid(True, alpha=0.3)
ax3.legend()

# ── 4. Distance to x* & Gradient Norm ───────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(steps, history["dist_to_xstar"], color="#9333EA", lw=1.8, label="||xₖ − x*||")
ax4_r = ax4.twinx()
ax4_r.plot(steps, history["grad_norm"], color="#F97316", lw=1.2,
           alpha=0.7, ls=":", label="||g(xₖ)||")
ax4.set_title("Distance to x*  &  Gradient Norm", fontweight="bold")
ax4.set_xlabel("Step k")
ax4.set_ylabel("||xₖ − x*||₂", color="#9333EA")
ax4_r.set_ylabel("||g(xₖ)||₂", color="#F97316")
ax4.grid(True, alpha=0.3)
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_r.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.savefig("/tmp/zo_gd_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to /tmp/zo_gd_curves.png")

## 7. Effect of Hyperparameters

Run the optimizer with several configurations and compare the loss curves.

In [ ]:
configs = [
    dict(lr=0.5,  mu=0.1,  batch_size=10,  label="lr=0.5, μ=0.1, B=10",  color="#2563EB"),
    dict(lr=0.5,  mu=0.01, batch_size=10,  label="lr=0.5, μ=0.01, B=10", color="#DC2626"),
    dict(lr=0.5,  mu=0.1,  batch_size=50,  label="lr=0.5, μ=0.1, B=50",  color="#16A34A"),
    dict(lr=0.1,  mu=0.1,  batch_size=10,  label="lr=0.1, μ=0.1, B=10",  color="#F97316"),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Hyperparameter Comparison (Zero-Order GD)", fontsize=13, fontweight="bold")

for cfg in configs:
    np.random.seed(0)
    hist, _ = zero_order_gd(
        x_init    = np.zeros(d),
        A         = A, y=y,
        lr        = cfg["lr"],
        mu        = cfg["mu"],
        batch_size= cfg["batch_size"],
        n_steps   = n_steps,
        lam       = lam,
        verbose   = False,
    )
    axes[0].plot(hist["loss"],        color=cfg["color"], lw=1.8, label=cfg["label"])
    shifted = np.array(hist["loss"]) - np.array(hist["loss"]).min() + 1e-12
    axes[1].semilogy(shifted,         color=cfg["color"], lw=1.8, label=cfg["label"])

for ax, title, ylabel in zip(
    axes,
    ["Training Loss", "Loss – min (log scale)"],
    ["f(xₖ)",          "f(xₖ) – f*  (log)"]
):
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Step k")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, which="both")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/hyperparam_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Comparison plot saved.")

## 8. Effect of Regularization

In [ ]:
lam_values = [0.0, 1e-3, 1e-2, 0.1]
colors     = ["#2563EB", "#16A34A", "#F97316", "#DC2626"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Effect of L2 Regularization λ  (lr=0.5, μ=0.1, B=20)",
             fontsize=13, fontweight="bold")

for lam_val, col in zip(lam_values, colors):
    np.random.seed(0)
    hist, _ = zero_order_gd(
        x_init    = np.zeros(d),
        A         = A, y=y,
        lr        = 0.5,
        mu        = 0.1,
        batch_size= 20,
        n_steps   = n_steps,
        lam       = lam_val,
        verbose   = False,
    )
    label = f"λ = {lam_val}"
    axes[0].plot(hist["loss"],  color=col, lw=1.8, label=label)
    axes[1].plot(hist["param_norm"], color=col, lw=1.8, label=label)

axes[0].set_title("Training Loss vs. Step",     fontweight="bold")
axes[0].set_xlabel("Step k"); axes[0].set_ylabel("f(xₖ)")
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].set_title("Parameter Norm vs. Step",    fontweight="bold")
axes[1].set_xlabel("Step k"); axes[1].set_ylabel("||xₖ||₂")
axes[1].grid(True, alpha=0.3); axes[1].legend()

plt.tight_layout()
plt.savefig("/tmp/regularization_effect.png", dpi=120, bbox_inches="tight")
plt.show()
print("Regularization plot saved.")

## 9. Summary & Discussion

| Concept | Role in this notebook |
|---|---|
| **Logistic Loss** | Smooth, convex objective on synthetic binary data |
| **Bias column** | Bias included via the appended `1`-column in $A$ |
| **L2 Regularization** | Controls parameter magnitude; trades off fit vs. complexity |
| **Random unit vectors** | Explore loss landscape via Gaussian $\to$ normalized samples |
| **Smoothing parameter $\mu$** | Controls the accuracy of the gradient estimate; smaller $\mu$ → better approximation but higher variance |
| **Batch size $B$** | More directions → lower-variance gradient estimate; higher cost per step |
| **Learning rate $\eta$** | Standard stepsize tradeoff: too large → diverges, too small → slow |

### Exercises

1. **Convergence rate**: How does the number of steps needed scale with $d$? Theory predicts $\mathcal{O}(d)$ overhead compared to first-order methods.
2. **Variance reduction**: Replace the forward-difference estimator with a **central-difference** version $\frac{f(x+\mu u) - f(x-\mu u)}{2\mu}$. Does convergence improve?
3. **Antithetic sampling**: For each $u_i$, also use $-u_i$ to halve the variance. Implement and compare.
4. **Dimension scaling**: Fix all other parameters and vary `dim` from 5 to 100. Plot final loss vs. `dim`.
5. **Adaptive $\mu$**: Decay $\mu_k = \mu_0 / \sqrt{k+1}$ and observe the effect on convergence.